In [17]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import matplotlib.pyplot as plt

# Load the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
data = pd.read_csv(url, delimiter=";")
dup_data = data

In [18]:
X = data.iloc[:, :-1].values  # Features (all except last column)
y = data.iloc[:, -1].values   # Target (wine quality score)

# One-hot encode the target labels
ohe = OneHotEncoder(sparse_output=False)
y_one_hot = ohe.fit_transform(y.reshape(-1, 1))

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [19]:
# Activation functions
def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def xavier_init(input_size, output_size):
    return np.random.randn(input_size, output_size) * np.sqrt(2. / (input_size + output_size))

def update_weights(W, b, grads, learning_rate):
    dW, db = grads
    W -= learning_rate * dW
    b -= learning_rate * db
    return W, b

In [20]:
# Generator model (with forward pass and weight updates)
def build_generator(input_size, hidden_size, feature_size, label_size):
    def forward(X, W1, b1, W2, b2, W3, b3):
        Z1 = np.dot(X, W1) + b1
        A1 = relu(Z1)

        # Feature generation
        Z2 = np.dot(A1, W2) + b2
        synthetic_features = Z2  # This will generate the feature part

        # Label generation
        Z3 = np.dot(A1, W3) + b3
        synthetic_labels = softmax(Z3)  # This will generate the label (softmax for multi-class)

        return synthetic_features, synthetic_labels, A1, Z1, Z3

    return forward

In [21]:
# Discriminator model (with forward pass and weight updates)
def build_discriminator(input_size, hidden_size, feature_size, label_size):
    # Forward pass
    def forward(X_features, X_labels, W1, b1, W2, b2):
        X = np.concatenate([X_features, X_labels], axis=1)  # Concatenate features and labels
        Z1 = np.dot(X, W1) + b1
        A1 = relu(Z1)

        Z2 = np.dot(A1, W2) + b2
        D_output = sigmoid(Z2)  # Output a value between 0 and 1 (real/fake)

        return D_output, A1, Z1

    return forward

In [22]:
# Set the input, hidden, and output sizes for the generator and discriminator
input_size = X_scaled.shape[1]  # Number of features (11)
hidden_size = 16  # Number of neurons in hidden layers
feature_size = X_scaled.shape[1]  # Number of features (11)
label_size = y_one_hot.shape[1]  # Number of unique wine quality classes (6)

# Initialize the Generator and Discriminator models
generator_model = build_generator(input_size, hidden_size, feature_size, label_size)
discriminator_model = build_discriminator(input_size, hidden_size, feature_size, label_size)

In [23]:
# Training function for the GAN
def train_gan(n_epochs=1000, batch_size=64, learning_rate=0.0002):
    # Initialize weights for generator and discriminator
    W1_g, b1_g, W2_g, b2_g, W3_g, b3_g = (xavier_init(input_size, hidden_size),
                                            np.zeros((1, hidden_size)),
                                            xavier_init(hidden_size, feature_size),
                                            np.zeros((1, feature_size)),
                                            xavier_init(hidden_size, label_size),
                                            np.zeros((1, label_size)))

    W1_d, b1_d, W2_d, b2_d = (xavier_init(input_size + label_size, hidden_size),
                               np.zeros((1, hidden_size)),
                               xavier_init(hidden_size, 1),
                               np.zeros((1, 1)))

    for epoch in range(n_epochs):
        # Generate synthetic data (noise)
        noise = np.random.randn(batch_size, input_size)
        X_fake, y_fake, A1_g, Z1_g, Z3_g = generator_model(noise, W1_g, b1_g, W2_g, b2_g, W3_g, b3_g)
        y_fake = np.eye(y_fake.shape[1])[np.argmax(y_fake, axis=1)]

        # Select a batch of real data
        indices = np.random.choice(X_scaled.shape[0], batch_size, replace=False)
        X_real_batch = X_scaled[indices]
        y_real_batch = y_one_hot[indices]

        # Train the discriminator on real and fake data
        D_real, A1_d, Z1_d = discriminator_model(X_real_batch, y_real_batch, W1_d, b1_d, W2_d, b2_d)
        D_fake, A1_d_fake, Z1_d_fake = discriminator_model(X_fake, y_fake, W1_d, b1_d, W2_d, b2_d)

        # Add label smoothing for real data (real labels close to 1, but not exactly 1)
        real_labels = np.ones((batch_size, 1)) * 0.9
        fake_labels = np.zeros((batch_size, 1))

        # Discriminator loss
        D_loss_real = -np.mean(np.log(D_real))  # Real loss
        D_loss_fake = -np.mean(np.log(1 - D_fake))  # Fake loss
        D_loss = D_loss_real + D_loss_fake

        # Gradients for discriminator
        D_real_grad = D_real - real_labels
        D_fake_grad = D_fake - fake_labels

        # Discriminator backpropagation
        dW1_d = np.dot(np.concatenate([X_real_batch, y_real_batch], axis=1).T, D_real_grad)
        db1_d = np.sum(D_real_grad, axis=0, keepdims=True)
        dW2_d = np.dot(A1_d.T, D_real_grad)
        db2_d = np.sum(D_real_grad, axis=0, keepdims=True)

        # Update discriminator weights
        W1_d, b1_d = update_weights(W1_d, b1_d, [dW1_d, db1_d], learning_rate)
        W2_d, b2_d = update_weights(W2_d, b2_d, [dW2_d, db2_d], learning_rate)

        # Generator loss (trying to fool the discriminator)
        D_fake_for_gen = discriminator_model(X_fake, y_fake, W1_d, b1_d, W2_d, b2_d)[0]
        G_loss = -np.mean(np.log(D_fake_for_gen))  # Fake loss

        # Generator backpropagation
        G_grad = D_fake_for_gen - np.ones_like(D_fake_for_gen)
        dW1_g = np.dot(noise.T, G_grad)
        db1_g = np.sum(G_grad, axis=0, keepdims=True)
        dW2_g = np.dot(A1_g.T, G_grad)
        db2_g = np.sum(G_grad, axis=0, keepdims=True)

        # Update generator weights
        W1_g, b1_g = update_weights(W1_g, b1_g, [dW1_g, db1_g], learning_rate)
        W2_g, b2_g = update_weights(W2_g, b2_g, [dW2_g, db2_g], learning_rate)
        W3_g, b3_g = update_weights(W3_g, b3_g, [dW2_g, db2_g], learning_rate)

        if epoch % 100 == 0:
            print(f"Epoch {epoch}/{n_epochs}, D Loss: {D_loss:.4f}, G Loss: {G_loss:.4f}")

    # Return the trained weights for the generator
    return W1_g, b1_g, W2_g, b2_g, W3_g, b3_g

# Train the GAN and get the weights for later use
W1_g, b1_g, W2_g, b2_g, W3_g, b3_g = train_gan(n_epochs=10000, batch_size=64, learning_rate=0.0002)


Epoch 0/10000, D Loss: 1.3164, G Loss: 1.2560
Epoch 100/10000, D Loss: 0.5016, G Loss: 1.5177
Epoch 200/10000, D Loss: 6.0172, G Loss: 0.0043
Epoch 300/10000, D Loss: 6.9512, G Loss: 0.0018
Epoch 400/10000, D Loss: 6.9624, G Loss: 0.0022
Epoch 500/10000, D Loss: 6.9599, G Loss: 0.0021
Epoch 600/10000, D Loss: 7.0286, G Loss: 0.0020
Epoch 700/10000, D Loss: 7.1997, G Loss: 0.0013
Epoch 800/10000, D Loss: 7.3086, G Loss: 0.0014
Epoch 900/10000, D Loss: 7.9681, G Loss: 0.0008
Epoch 1000/10000, D Loss: 7.7625, G Loss: 0.0010
Epoch 1100/10000, D Loss: 8.1917, G Loss: 0.0008
Epoch 1200/10000, D Loss: 8.3601, G Loss: 0.0006
Epoch 1300/10000, D Loss: 8.2356, G Loss: 0.0005
Epoch 1400/10000, D Loss: 7.8787, G Loss: 0.0009
Epoch 1500/10000, D Loss: 8.0871, G Loss: 0.0007
Epoch 1600/10000, D Loss: 8.4120, G Loss: 0.0005
Epoch 1700/10000, D Loss: 8.5601, G Loss: 0.0005
Epoch 1800/10000, D Loss: 8.7148, G Loss: 0.0006
Epoch 1900/10000, D Loss: 8.9480, G Loss: 0.0005
Epoch 2000/10000, D Loss: 8.7568

In [24]:
batch_size = 100  # Number of new synthetic samples
noise = np.random.randn(batch_size, input_size)
generated_features, generated_labels, A1_g, Z1_g, Z3_g = generator_model(noise, W1_g, b1_g, W2_g, b2_g, W3_g, b3_g)

# Print the generated synthetic data
print("Generated Features:\n", generated_features)
print("Generated Labels:\n", np.argmax(generated_labels, axis=1))

correct_labels = np.argmax(generated_labels, axis=1).reshape(-1, 1)

# Combine generated features and labels into one array
data = np.hstack((generated_features, correct_labels))  # Add labels as last column

# Create a DataFrame
df = pd.DataFrame(data)

# Assign the column names to the DataFrame
df.columns = dup_data.columns

# Save the DataFrame to a CSV file
df.to_csv('generated_data.csv', index=False)  # Save with column names and no index

print("Generated data saved to 'generated_data.csv'")


Generated Features:
 [[18.63012526 20.21830651 16.77038781 ... 17.41914796 19.6448108
  19.02735179]
 [23.82124054 24.2030236  19.93371966 ... 23.48533371 23.90618058
  23.00403561]
 [29.313016   28.53096147 26.02260292 ... 29.82391481 29.79640978
  27.8681712 ]
 ...
 [22.10270174 22.80962655 20.71112153 ... 22.76033851 23.4489935
  23.46134059]
 [23.70846908 23.7265654  22.25579175 ... 24.55613525 24.95240217
  24.21772595]
 [21.15239774 20.57708951 18.54356496 ... 21.71187329 20.96673847
  19.77484481]]
Generated Labels:
 [2 2 1 1 2 1 1 3 5 5 2 5 2 3 3 3 4 1 1 4 2 1 2 3 4 3 4 2 4 1 2 2 1 1 2 2 4
 4 1 3 5 1 5 4 2 4 2 4 2 4 4 2 1 2 2 1 3 2 4 1 4 1 4 1 2 4 4 2 2 3 4 2 2 2
 5 2 1 4 2 1 1 5 1 5 5 4 2 3 2 1 1 4 4 4 5 1 5 1 1 4]
Generated data saved to 'generated_data.csv'
